# E-commerce: Bronze to Silver with Structured Streaming

**Context:** CSV invoice files arrive in HDFS. Convert newly discovered files into cleaned sales records in a Delta directory.

`HDFS CSV /bronze/ecomm ? validation and Amount ? HDFS Delta /silver/ecomm`

This pipeline uses Spark's standard **CSV file streaming source**. The schema is explicit, transformations are lazy, and `.start()` begins micro-batch execution. No cloud ingestion service is involved.

## Environment and storage assumptions

Examples target Spark 3.5.7 with open-source Delta Lake 3.3.2. Install matching packages in the notebook environment before creating Spark: `pip install pyspark==3.5.7 delta-spark==3.3.2`. Delta's JVM dependencies must also be available; the session helper resolves them through Maven on first use.

HDFS is available at `hdfs://localhost:9000`. This is a local Spark deployment: in a distributed deployment, `localhost` would point to each machine separately and must be replaced with a reachable NameNode hostname. HDFS paths must be readable/writable by the Spark user.

Delta is a storage format, not a cloud service. We access Delta tables by HDFS path; no Hive service, persistent metastore, or named database is needed. `DeltaCatalog` provides Spark integration and does not require Hive. Use a fresh kernel if Spark was already created without Delta extensions.

Code is supplied for explanation and adaptation; it has not been run against Spark or HDFS. Timestamp examples are timezone-free; UTC is the explicit interpretation used here. Change it to the source's documented timezone when necessary.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

# Configure Delta before creating the session; no Hive metastore is used.
builder = (SparkSession.builder.appName("EcommHDFSStreaming")
    .master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalogImplementation", "in-memory")
    .config("spark.sql.shuffle.partitions", "2"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

## Paths and file arrival

Input, output and checkpoint directories serve different purposes. A checkpoint tracks processed input and recovery progress; it is not the output data.

Create `/bronze/ecomm` in HDFS before starting. Publish complete CSV files atomically under unique names, for example by moving a completed file from a staging directory. Do not edit a file after ingestion. `maxFilesPerTrigger` limits admitted files, not records.

The expected header order is: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country.

In [ ]:
HDFS_ROOT = "hdfs://localhost:9000"
ECOMM_BRONZE_FILES = HDFS_ROOT + "/bronze/ecomm"
ECOMM_SILVER_FILES = HDFS_ROOT + "/silver/ecomm"
CHECKPOINT_LOCATION = HDFS_ROOT + "/checkpoints/ecomm/bronze_to_silver"

## Read raw fields with a stable schema

Read fields as strings first so date/numeric conversion is explicit. Identifiers such as StockCode and CustomerID are labels, not measures. Keeping them as strings preserves their spelling.

`header=true` skips the header; `enforceSchema=false` checks header compatibility. `FAILFAST` stops on structurally malformed CSV records rather than silently accepting a damaged row. A well-formed row containing an invalid number is handled separately during conversion.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

raw_columns = ["InvoiceNo", "StockCode", "Description", "Quantity",
               "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
ecomm_schema = StructType([StructField(name, StringType(), True) for name in raw_columns])

bronze_df = (spark.readStream.format("csv")
    .option("header", "true")
    .option("enforceSchema", "false")
    .option("mode", "FAILFAST")
    .option("maxFilesPerTrigger", 5)
    .schema(ecomm_schema)
    .load(ECOMM_BRONZE_FILES))

bronze_df.printSchema()  # Inspects the schema; does not start the query.
print(bronze_df.isStreaming)

## Convert once, then validate

**Date context:** the supplied code assumed `yyyy-MM-dd HH:mm:ss`. Retail CSV exports may instead use `M/d/yyyy H:mm`. The example accepts those two formats explicitly; add another only after inspecting the actual source contract. `try_to_timestamp` returns null for invalid input.

Use decimal currency values instead of binary floating-point amounts. Invalid/overflowing numeric conversions become null through `try_cast` and are rejected below. CustomerID remains a trimmed string.

In [ ]:
trimmed_df = bronze_df.select([
    F.when(F.length(F.trim(F.col(name))) > 0, F.trim(F.col(name)))
     .otherwise(F.lit(None).cast("string")).alias(name)
    for name in raw_columns
])

typed_df = (trimmed_df
    .withColumn("Quantity", F.expr("try_cast(Quantity as int)"))
    .withColumn("UnitPrice", F.expr("try_cast(UnitPrice as decimal(18, 2))"))
    .withColumn("InvoiceDate", F.coalesce(
        F.try_to_timestamp("InvoiceDate", F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp("InvoiceDate", F.lit("M/d/yyyy H:mm")))))

## Define the sales-only cleaning policy

Keep rows with valid invoice/product/customer/country/time fields, positive quantity and a non-negative price. Exclude invoices beginning with C (case-insensitive), representing cancellations/returns.

A zero quantity contributes no sale and is excluded; zero-priced items are retained and contribute zero revenue. Description is optional. This is **sales excluding returns**, not net revenue after refunds.

Example: 3 units at 2.50 produce Amount = 7.50. A cancelled invoice or invalid date is excluded. Rejected rows are filtered in this teaching pipeline; a production design should record rejection reasons in a separate quarantine output. No business-event deduplication is implied.

In [ ]:
cleaned_df = (typed_df
    .filter(F.col("InvoiceNo").isNotNull())
    .filter(~F.upper(F.col("InvoiceNo")).startswith("C"))
    .filter(F.col("StockCode").isNotNull())
    .filter(F.col("CustomerID").isNotNull())
    .filter(F.col("Country").isNotNull())
    .filter(F.col("InvoiceDate").isNotNull())
    .filter(F.col("Quantity") > 0)
    .filter(F.col("UnitPrice") >= 0)
    .withColumn("year", F.year("InvoiceDate"))
    .withColumn("month", F.month("InvoiceDate"))
    .withColumn("day", F.dayofmonth("InvoiceDate")))

with_amount_df = cleaned_df.withColumn("Amount", F.col("Quantity") * F.col("UnitPrice"))
with_amount_df.printSchema()  # Still only a query definition.

## Write new Silver rows

Append fits this stateless transformation: each accepted invoice line becomes a new Silver row. Delta stores its data and transaction log directly in HDFS.

A one-minute trigger controls scheduling, not the event-time grouping. If a batch takes longer than one minute, the next starts after it completes. Date partitioning follows the existing lesson; many small date partitions/files may be inefficient for small datasets.

Keep this pipeline's checkpoint across compatible restarts. Do not reuse it for a different source or schema. Rerunning `.start()` while the prior query is active must not create a second writer with the same checkpoint.

In [ ]:
if "silver_query" in globals() and silver_query.isActive:
    raise RuntimeError("Stop the existing Silver query before starting another one.")

silver_query = (with_amount_df.writeStream
    .format("delta")
    .outputMode("append")
    .partitionBy("year", "month", "day")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(processingTime="1 minute")
    .queryName("ecomm_bronze_to_silver")
    .start(ECOMM_SILVER_FILES))

## Inspect results with ordinary Spark APIs

A streaming DataFrame cannot use a batch `show()` action directly. Inspect its schema/plan, use query progress, or read already committed Silver output as a **bounded** DataFrame.

The sample below waits for no data: run it after at least one successful Silver commit. If there are no valid input rows yet, output may be empty or not initialized. No notebook-specific display API is needed.

In [ ]:
print(silver_query.status)
print(silver_query.lastProgress)  # May be None before the first completed trigger.

# Optional bounded preview after Silver exists:
# spark.read.format("delta").load(ECOMM_SILVER_FILES).show(10, truncate=False)

## Stop, wait and recover

`stop()` stops this query but leaves Spark available. `awaitTermination(10)` waits up to ten seconds; its timeout does not stop the query. An indefinite wait belongs in a dedicated streaming application, not an automatic notebook reading flow.

The same invoice row delivered in a differently named file is new source input and can be counted again. A file checkpoint tracks files, not a unique invoice-line business key.

In [ ]:
# Use explicitly when finished with this query:
# silver_query.stop()

# Optional wait: returns False on timeout; failures raise an exception.
# silver_query.awaitTermination(10)

## References

- [Spark 3.5.7 Structured Streaming guide](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html)
- [Delta Lake / Spark compatibility](https://docs.delta.io/releases/)
- [Delta Lake session setup](https://docs.delta.io/quick-start/)
- [Delta streaming behavior](https://docs.delta.io/delta-streaming/)

These are adapted copies of the supplied notebooks. The originals in Downloads are unchanged. Saved outputs and platform-specific notebook metadata have been removed.